In [1]:
# Step 1: Import library and load processed dataset
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\Desktop\Supply Chain Analytics\cleaned_supply_chain_dataset1.csv")
df.head()

,date,sku_id,warehouse_id,supplier_id,region,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,...,demand_forecast,total_revenue,total_cost,profit,profit_margin,below_reorder_flag,lost_sales_units,lost_revenue,forecast_error,abs_forecast_error
0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,...,8.52,204.80,139.50,65.30,0.318848,0,0.0,0.0,1.48,1.48
1,2024-01-02,SKU_1,WH_1,SUP_8,West,17,575,14,379,0,...,18.63,348.16,237.15,111.01,0.318848,0,0.0,0.0,-1.63,1.63
2,2024-01-03,SKU_1,WH_1,SUP_8,North,35,540,14,379,0,...,39.62,716.80,488.25,228.55,0.318848,0,0.0,0.0,-4.62,4.62
3,2024-01-04,SKU_1,WH_1,SUP_8,South,24,516,14,379,0,...,19.43,491.52,334.80,156.72,0.318848,0,0.0,0.0,4.57,4.57
4,2024-01-05,SKU_1,WH_1,SUP_8,West,21,495,14,379,0,...,18.70,430.08,292.95,137.13,0.318848,0,0.0,0.0,2.30,2.30


In [3]:
df.shape

(91250, 24)

In [4]:
df.isnull().sum()

date                       0
sku_id                     0
warehouse_id               0
supplier_id                0
region                     0
units_sold                 0
inventory_level            0
supplier_lead_time_days    0
reorder_point              0
order_quantity             0
unit_cost                  0
unit_price                 0
promotion_flag             0
stockout_flag              0
demand_forecast            0
total_revenue              0
total_cost                 0
profit                     0
profit_margin              0
below_reorder_flag         0
lost_sales_units           0
lost_revenue               0
forecast_error             0
abs_forecast_error         0
dtype: int64

In [9]:
# Step 2: Extract unique dates and build granular Date Dimension (dim_date)
df['date'] = pd.to_datetime(df['date'])
dates = pd.DataFrame({'date': df['date'].unique()}).sort_values('date').reset_index(drop=True)
dim_date = pd.DataFrame({
    'date': dates['date'],
    'year': dates['date'].dt.year,
    'quarter': dates['date'].dt.quarter,
    'month': dates['date'].dt.month,
    'day' : dates['date'].dt.day,
    'month_name': dates['date'].dt.month_name(),
    'day_of_week': dates['date'].dt.dayofweek + 1,
    'day_name': dates['date'].dt.day_name()
})

In [13]:
dim_date.head()

,date,year,quarter,month,day,month_name,day_of_week,day_name
0,2024-01-01,2024,1,1,1,January,1,Monday
1,2024-01-02,2024,1,1,2,January,2,Tuesday
2,2024-01-03,2024,1,1,3,January,3,Wednesday
3,2024-01-04,2024,1,1,4,January,4,Thursday
4,2024-01-05,2024,1,1,5,January,5,Friday


In [17]:
# Step 3: Export Date Dimension table to CSV
dim_date.to_csv("dim_date.csv" , index = False)

In [3]:
# Step 4: Structure daily Fact Table (fact_inventory_daily) with 24 columns
fact_cols = [
    'date', 'sku_id', 'warehouse_id', 'supplier_id', 'unit_cost', 'unit_price',
    'units_sold', 'inventory_level', 'supplier_lead_time_days',
    'reorder_point', 'order_quantity', 'promotion_flag', 'stockout_flag',
    'demand_forecast', 'total_revenue', 'total_cost', 'profit',
    'profit_margin', 'below_reorder_flag', 'lost_sales_units',
    'lost_revenue', 'forecast_error', 'abs_forecast_error' , 'region'
]

fact_inventory_daily = df[fact_cols].copy()

In [4]:
fact_inventory_daily.head()

,date,sku_id,warehouse_id,supplier_id,unit_cost,unit_price,units_sold,inventory_level,supplier_lead_time_days,reorder_point,...,total_revenue,total_cost,profit,profit_margin,below_reorder_flag,lost_sales_units,lost_revenue,forecast_error,abs_forecast_error,region
0,2024-01-01,SKU_1,WH_1,SUP_8,13.95,20.48,10,592,14,379,...,204.80,139.50,65.30,0.318848,0,0.0,0.0,1.48,1.48,West
1,2024-01-02,SKU_1,WH_1,SUP_8,13.95,20.48,17,575,14,379,...,348.16,237.15,111.01,0.318848,0,0.0,0.0,-1.63,1.63,West
2,2024-01-03,SKU_1,WH_1,SUP_8,13.95,20.48,35,540,14,379,...,716.80,488.25,228.55,0.318848,0,0.0,0.0,-4.62,4.62,North
3,2024-01-04,SKU_1,WH_1,SUP_8,13.95,20.48,24,516,14,379,...,491.52,334.80,156.72,0.318848,0,0.0,0.0,4.57,4.57,South
4,2024-01-05,SKU_1,WH_1,SUP_8,13.95,20.48,21,495,14,379,...,430.08,292.95,137.13,0.318848,0,0.0,0.0,2.30,2.30,West


In [5]:
# Step 5: Export Fact Table to CSV for PostgreSQL database import
fact_inventory_daily.to_csv("fact_inventory_daily.csv" , index = False)